In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# 1. WATERMARK: Find the last date we ingested
try:
    # We check Silver for the last date because it's our "truth"
    last_date_row = spark.sql("SELECT MAX(date) FROM nasa_project.silver.asteroids_cleaned").collect()[0][0]
    # NASA API end_date is inclusive, so we start the day AFTER our last success
    start_date = (last_date_row + timedelta(days=1)) if last_date_row else (datetime.now() - timedelta(days=7)).date()
except:
    start_date = (datetime.now() - timedelta(days=7)).date()

end_date = datetime.now().date()

# 2. FETCH DATA: Only if start_date is not in the future
if start_date <= end_date:
    API_KEY = "Bafqfo3mdbUeKMQn2HvgrpnB8O31CEAKQRvWio61"
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    
    response = requests.get(url)
    if response.status_code == 200:
        raw_json = response.text
        # Add a timestamp so we can track the ingestion batch
        ingestion_ts = datetime.now()
        
        # Save to Bronze (Append mode is critical for daily runs)
        bronze_df = spark.createDataFrame([(raw_json, ingestion_ts)], ["raw_json", "ingested_at"])
        bronze_df.write.mode("append").saveAsTable("nasa_project.bronze.asteroids_raw")
        print(f"Ingested data from {start_date} to {end_date}")
    else:
        print(f"NASA API Error: {response.status_code}")
else:
    print("No new dates to fetch today!")